# Team Research Bot Build — Practical Notebook

This practical builds on the earlier role-design and delegation work. The goal is to turn roles into a working collaboration protocol.

You will build and inspect a four-agent research team:

**Researcher → Writer → Fact-Checker → Editor**

The main learning objective is not "how to write four prompts." The real objective is to see how reliable agent teams depend on structured messages, Pydantic validation, shared state, and traceable handoffs.

## Practical map

1. Load the local source pack  
2. Inspect schemas as agent-to-agent contracts  
3. Create a serialized message envelope  
4. Run each agent independently  
5. Run the complete orchestrated pipeline  
6. Inspect the saved JSON trace  
7. Break the system deliberately and observe validation  
8. Try extension exercises

In [15]:
import os
from pathlib import Path
import json
from dotenv import load_dotenv

load_dotenv(override=True)

# TRAINER NOTE: Keep mock mode on for the first classroom run.
# It proves the architecture without API-key friction or cost.
os.environ.setdefault("USE_MOCK_LLM", "false")

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)
print("Mock mode:", os.environ.get("USE_MOCK_LLM"))

Project root: E:\BIA\team_research_bot\team_research_bot
Mock mode: false


## Load the local source pack

The Researcher will use these documents as its evidence base. A local source pack keeps the session focused on **agent collaboration** instead of live web search setup.

In [16]:
from source_loader import load_source_pack

documents = load_source_pack("data/source_pack")
[(doc.source_id, doc.title) for doc in documents]

[('SRC-001', 'AI Agents in Customer Support Operations'),
 ('SRC-002', 'Responsible AI in Hiring Workflows'),
 ('SRC-003', 'Healthcare Administrative Agents'),
 ('SRC-004', 'Multi-Agent Governance Patterns')]

## Inspect schemas as contracts

A schema is an API contract between two agents. It says: "the next agent can rely on these fields being present and typed correctly."

The most important schema is `AgentMessage`, because every handoff moves through this envelope.

In [17]:
from schemas import (
    AgentMessage,
    AgentRole,
    MessageType,
    ResearchReport,
    DraftReport,
    FactCheckReport,
    FinalReport,
)

AgentMessage.model_json_schema()

{'$defs': {'AgentRole': {'description': 'Named roles in the research team.',
   'enum': ['orchestrator', 'researcher', 'writer', 'fact_checker', 'editor'],
   'title': 'AgentRole',
   'type': 'string'},
  'MessageType': {'description': 'Allowed types of messages passed between agents.',
   'enum': ['task', 'result', 'critique', 'final'],
   'title': 'MessageType',
   'type': 'string'}},
 'additionalProperties': False,
 'description': 'Serializable envelope for any inter-agent handoff.',
 'properties': {'trace_id': {'title': 'Trace Id', 'type': 'string'},
  'sender': {'$ref': '#/$defs/AgentRole'},
  'receiver': {'$ref': '#/$defs/AgentRole'},
  'message_type': {'$ref': '#/$defs/MessageType'},
  'task': {'title': 'Task', 'type': 'string'},
  'payload': {'additionalProperties': True,
   'title': 'Payload',
   'type': 'object'},
  'confidence': {'maximum': 1,
   'minimum': 0,
   'title': 'Confidence',
   'type': 'number'},
  'created_at': {'title': 'Created At', 'type': 'string'}},
 'requir

## Build a valid serialized message

This message has a sender, receiver, task, payload, confidence score, and trace ID. In production, this is the difference between a debuggable workflow and a long ambiguous transcript.

In [18]:
from schemas import EvidenceItem, ResearchFinding, ResearchReport, make_message

sample_report = ResearchReport(
    topic="AI agents in customer support",
    findings=[
        ResearchFinding(
            claim="AI support agents work best when paired with escalation paths.",
            evidence=[
                EvidenceItem(
                    source_id="SRC-001",
                    title="AI Agents in Customer Support Operations",
                    snippet="The rollout improved only after the team added an escalation rule for unresolved or high-frustration conversations.",
                    relevance_score=0.92,
                )
            ],
            confidence=0.84,
            limitations="This is based on the bundled classroom source pack, not live market research.",
        ),
        ResearchFinding(
            claim="Structured handoffs improve traceability in multi-agent workflows.",
            evidence=[
                EvidenceItem(
                    source_id="SRC-004",
                    title="Multi-Agent Governance Patterns",
                    snippet="Each handoff should include sender, receiver, task, payload, confidence, and trace ID.",
                    relevance_score=0.88,
                )
            ],
            confidence=0.81,
            limitations="The source is a teaching note, so it should be validated against production logs later.",
        ),
    ],
    unresolved_questions=["Would live support metrics confirm this pattern today?"],
    overall_confidence=0.82,
)

message = make_message(
    sender=AgentRole.RESEARCHER,
    receiver=AgentRole.WRITER,
    message_type=MessageType.RESULT,
    task="Use these findings to draft a research brief.",
    payload_model=sample_report,
    confidence=sample_report.overall_confidence,
)

print(message.model_dump_json(indent=2))

{
  "trace_id": "msg_4972d694",
  "sender": "researcher",
  "receiver": "writer",
  "message_type": "result",
  "task": "Use these findings to draft a research brief.",
  "payload": {
    "topic": "AI agents in customer support",
    "findings": [
      {
        "claim": "AI support agents work best when paired with escalation paths.",
        "evidence": [
          {
            "source_id": "SRC-001",
            "title": "AI Agents in Customer Support Operations",
            "snippet": "The rollout improved only after the team added an escalation rule for unresolved or high-frustration conversations.",
            "relevance_score": 0.92
          }
        ],
        "confidence": 0.84,
        "limitations": "This is based on the bundled classroom source pack, not live market research."
      },
      {
        "claim": "Structured handoffs improve traceability in multi-agent workflows.",
        "evidence": [
          {
            "source_id": "SRC-004",
            "title":

## Validation catches bad handoffs early

The next cell intentionally creates a bad message. The confidence score is outside the allowed 0–1 range, so Pydantic should reject it before another agent consumes the payload.

In [19]:
try:
    bad_message = AgentMessage(
        trace_id="bad_demo",
        sender=AgentRole.RESEARCHER,
        receiver=AgentRole.WRITER,
        message_type=MessageType.RESULT,
        task="This should fail.",
        payload={},
        confidence=1.7,
    )
except Exception as exc:
    print("Validation caught the bad handoff:")
    print(exc)

Validation caught the bad handoff:
1 validation error for AgentMessage
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## Create the LLM client

The same code supports two modes:

- `USE_MOCK_LLM=true`: deterministic no-cost classroom mode
- `USE_MOCK_LLM=false`: real OpenAI calls using `OPENAI_API_KEY`

Cost note: the real run is designed to stay well under USD $0.50 with the bundled source pack and `gpt-4o-mini`.

In [20]:
from llm import StructuredLLMClient

llm_client = StructuredLLMClient()
print("Using model:", llm_client.model_name)
print("Mock mode:", llm_client.use_mock)

Using model: gpt-4o
Mock mode: False


## Run the Researcher agent

The Researcher does not write a final answer. It extracts evidence-backed findings and preserves source references for downstream review.

In [21]:
from agents.researcher import ResearcherAgent

query = "What should an organization consider before using AI agents in customer support?"

researcher = ResearcherAgent(llm_client)
research_report = researcher.run(query, documents)
print(research_report.model_dump_json(indent=2))

{
  "topic": "Considerations for Using AI Agents in Customer Support",
  "findings": [
    {
      "claim": "AI agents in customer support should be used primarily for triage and not full automation to improve operational efficiency.",
      "evidence": [
        {
          "source_id": "SRC-001",
          "title": "AI Agents in Customer Support Operations",
          "snippet": "The strongest operational gain came from triage rather than full automation. The AI assistant summarized the issue, collected account context, and suggested likely resolution paths.",
          "relevance_score": 0.9
        }
      ],
      "confidence": 0.9,
      "limitations": "The evidence is based on a single telecom company's experience and may not generalize to all industries or customer support scenarios."
    },
    {
      "claim": "Implementing a human escalation path is crucial for handling unresolved or complex customer queries effectively.",
      "evidence": [
        {
          "source_id":

## Run the Writer agent

The Writer turns structured findings into a readable draft. The Writer should not invent claims outside the Researcher's evidence.

In [30]:
research_report.model_dump_json(indent=2)

'{\n  "topic": "Considerations for Using AI Agents in Customer Support",\n  "findings": [\n    {\n      "claim": "AI agents in customer support should be used primarily for triage and not full automation to improve operational efficiency.",\n      "evidence": [\n        {\n          "source_id": "SRC-001",\n          "title": "AI Agents in Customer Support Operations",\n          "snippet": "The strongest operational gain came from triage rather than full automation. The AI assistant summarized the issue, collected account context, and suggested likely resolution paths.",\n          "relevance_score": 0.9\n        }\n      ],\n      "confidence": 0.9,\n      "limitations": "The evidence is based on a single telecom company\'s experience and may not generalize to all industries or customer support scenarios."\n    },\n    {\n      "claim": "Implementing a human escalation path is crucial for handling unresolved or complex customer queries effectively.",\n      "evidence": [\n        {\n

In [22]:
from agents.writer import WriterAgent

writer = WriterAgent(llm_client)
draft_report = writer.run(query, research_report)
print(draft_report.model_dump_json(indent=2))

{
  "title": "Key Considerations for Implementing AI Agents in Customer Support",
  "executive_summary": "Organizations looking to implement AI agents in customer support should focus on using these systems primarily for triage to enhance operational efficiency. It is crucial to establish a human escalation path for unresolved or complex queries and to ensure robust privacy controls to protect sensitive customer information. Clear ownership and policy guidelines are also necessary to manage AI-generated responses effectively. While these strategies can improve customer support operations, uncertainties remain regarding the handling of mixed-intent queries and the implementation of effective privacy controls.",
  "sections": [
    {
      "heading": "Optimizing AI Agents for Triage in Customer Support",
      "content": "AI agents should be primarily used for triage rather than full automation to enhance operational efficiency in customer support. By summarizing issues, collecting accou

## Run the Fact-Checker agent

The Fact-Checker maps the draft back to evidence. This is Reflexion in a team context: critique is separated from generation and returned as a structured report.

In [23]:
from agents.fact_checker import FactCheckerAgent

fact_checker = FactCheckerAgent(llm_client)
fact_check_report = fact_checker.run(draft_report, research_report)
print(fact_check_report.model_dump_json(indent=2))

{
  "checks": [
    {
      "claim": "AI agents should be primarily used for triage rather than full automation to enhance operational efficiency in customer support.",
      "verdict": "supported",
      "evidence_refs": [
        "SRC-001"
      ],
      "issue": "The claim is supported by evidence from a single telecom company's experience.",
      "recommendation": "Clarify that the evidence is based on a single telecom company's experience and may not be generalizable."
    },
    {
      "claim": "Implementing a human escalation path is essential for effectively handling unresolved or complex customer queries.",
      "verdict": "supported",
      "evidence_refs": [
        "SRC-001"
      ],
      "issue": "The claim is supported by evidence, but the specific escalation strategy may vary.",
      "recommendation": "Mention that the specific escalation strategy may vary depending on the complexity of issues and AI capabilities."
    },
    {
      "claim": "AI systems must includ

## Run the Editor agent

The Editor produces the final answer. It should preserve caveats instead of hiding uncertainty.

In [24]:
from agents.editor import EditorAgent

editor = EditorAgent(llm_client)
final_report = editor.run(draft_report, fact_check_report)
print(final_report.model_dump_json(indent=2))

{
  "title": "Key Considerations for Implementing AI Agents in Customer Support",
  "final_answer": "Organizations aiming to integrate AI agents into customer support should prioritize using these systems for triage to improve operational efficiency. It is essential to establish a human escalation path for complex queries and ensure robust privacy controls to protect sensitive customer information. Clear ownership and policy guidelines are also necessary to manage AI-generated responses effectively. While these strategies can enhance customer support operations, uncertainties remain regarding the handling of mixed-intent queries and the implementation of effective privacy controls. The insights provided are primarily based on a single telecom company's experience, which may not be universally applicable.",
  "key_takeaways": [
    "AI agents are most effective when used for triage rather than full automation, as evidenced by a telecom company's experience.",
    "Human escalation paths

## Run the full orchestrated pipeline

Now the orchestrator coordinates all agents, records inter-agent messages, and saves a trace JSON file.

In [26]:
from orchestrator import ResearchBotOrchestrator

orchestrator = ResearchBotOrchestrator(
    llm_client=llm_client,
    documents=documents,
    trace_dir="traces",
    max_revisions=1,
)

state = orchestrator.run(query)

print("Run ID:", state.run_id)
print("Status:", state.status)
print("Messages:", len(state.messages))
print("Sources consulted:", state.source_ids_consulted)
print("\nFinal answer preview:\n")
print(state.final_report.final_answer[:1000] if state.final_report else state.errors)

Run ID: run_31be005a63
Status: completed
Messages: 6
Sources consulted: ['SRC-001']

Final answer preview:

Organizations looking to implement AI agents in customer support should consider a layered design approach that emphasizes triage and summarization to enhance efficiency. This approach allows AI agents to handle initial tasks, potentially improving consistency and response times. However, these insights are primarily derived from a single case study in the telecom industry, highlighting the need for further research to validate these findings across different sectors. Additionally, clear ownership of AI-generated responses is essential to avoid policy-sensitive errors, and strong privacy controls are necessary to protect customer information. The current evidence lacks detailed strategies for privacy control implementation, necessitating further exploration to develop comprehensive solutions and ensure compliance with evolving data protection regulations.


## Inspect the message table

Each row below is one serialized handoff. This is the operational view of collaboration.

In [27]:
import pandas as pd

message_rows = ResearchBotOrchestrator.pretty_message_table(state)
pd.DataFrame(message_rows)

,trace_id,sender,receiver,type,task
0,msg_2e798a09,researcher,writer,result,Use these evidence-backed findings to draft a ...
1,msg_d628a1e4,writer,fact_checker,result,Check this draft against the Researcher evidence.
2,msg_3fe9921e,fact_checker,writer,critique,Revise the draft if needed; otherwise prepare ...
3,msg_6a969e0a,writer,fact_checker,result,Check the revised draft against the Researcher...
4,msg_bdf68aca,fact_checker,editor,critique,Use this final check to produce the final answer.
5,msg_551e0e2c,editor,orchestrator,final,Return the final answer to the user.


## Inspect the saved trace

The trace is the run-level memory: user query, messages, intermediate outputs, revision count, and final report.

In [28]:
trace_path = Path("traces") / f"{state.run_id}.json"
trace = json.loads(trace_path.read_text(encoding="utf-8"))
print("Trace file:", trace_path)
print("Top-level keys:", list(trace.keys()))
print("First message payload keys:", list(trace["messages"][0]["payload"].keys()))

Trace file: traces\run_31be005a63.json
Top-level keys: ['run_id', 'user_query', 'source_ids_consulted', 'messages', 'research_report', 'draft_report', 'fact_check_report', 'final_report', 'revision_count', 'max_revisions', 'status', 'errors']
First message payload keys: ['topic', 'findings', 'unresolved_questions', 'overall_confidence']


## Debugging drill: introduce a weak handoff

This cell modifies the draft to reference an unknown source ID. In mock mode, the Fact-Checker will detect the mismatch and ask for revision.

In [29]:
tampered_draft = draft_report.model_copy(deep=True)
tampered_draft.source_ids_used.append("SRC-999")

tampered_check = fact_checker.run(tampered_draft, research_report)
print(tampered_check.model_dump_json(indent=2))

{
  "checks": [
    {
      "claim": "AI agents should be primarily used for triage rather than full automation to enhance operational efficiency in customer support.",
      "verdict": "supported",
      "evidence_refs": [
        "SRC-001"
      ],
      "issue": "The claim is based on evidence from a single telecom company.",
      "recommendation": "Clarify that the recommendation is based on a specific case and may not apply universally across all industries."
    },
    {
      "claim": "Implementing a human escalation path is essential for effectively handling unresolved or complex customer queries.",
      "verdict": "supported",
      "evidence_refs": [
        "SRC-001"
      ],
      "issue": "The claim is well-supported but lacks detail on specific escalation strategies.",
      "recommendation": "Mention that specific escalation strategies may vary depending on the complexity of the queries and the capabilities of the AI system."
    },
    {
      "claim": "AI systems mus

## Exercise 1 — Add a new source

Add a new markdown file to `data/source_pack/`, then reload the source pack and run the Researcher again.

Hint: keep the file focused. A source with 3–5 short paragraphs is enough.

## Exercise 2 — Add a Compliance Reviewer

Create a fifth agent between Fact-Checker and Editor.

Suggested output schema:

```python
class ComplianceReview(BaseModel):
    approved: bool
    flagged_risks: list[str]
    required_changes: list[str]
```

Think about what input it should receive: draft only, fact-check report only, or both?

## Exercise 3 — Change the stop condition

Increase `max_revisions` from 1 to 2. Then inspect the trace and compare:

- How many messages were passed?
- Did the final answer improve?
- Did the extra pass justify the extra cost?

## Closing summary

A practical multi-agent system needs more than roles.

- **Prompts** define behavior.
- **Schemas** define handoffs.
- **State** defines what the run remembers.
- **Critique loops** improve quality.
- **Trace logs** make the system debuggable.

That is the foundation for production-style agent collaboration.